In [1]:
import scipy.io as sio
import numpy as np
import jax.numpy as jnp
import jax
#data_dict = mat73.loadmat('data.mat')
import matplotlib.pyplot as plt
import math
import os
import matplotlib
from functools import partial

def gett_all(ijlm, A):
    i, j, l, m = list(ijlm)
    pexp = i+'a,'+j+'a,'+l+'b,'+m+'b->'
    qexp = i+'a,'+j+'a,'+l+'a,'+m+'a->'
    pval = jnp.einsum(pexp, A, A, A, A)
    pqval = pval - jnp.einsum(qexp, A, A, A, A)
    return pval,pqval

@partial(jax.jit, static_argnames=['P','Q'])
def getest_all(A,P,Q):
    nf = jnp.sqrt(P)*jnp.sqrt(Q)
    # Wrap *all* the following in a single shared_intermediates context
    #with shared_intermediates():
    t1,t1d = gett_all('ijji', A/nf)
    t2,t2d = gett_all('iiii', A/nf)
    t3,t3d = gett_all('ijjj', A/nf)
    #t4,t4d = gett_all('iiij', A/nf)#<
    t5,t5d = gett_all('ijjl', A/nf)
    t6,t6d = gett_all('iijj', A/nf)
    t7,t7d = gett_all('iijl', A/nf)
    #t8,t8d = gett_all('ijll', A/nf)#<
    t9,t9d = gett_all('ijlm', A/nf)

    f1 = P / (P - 2)
    f2 = 2 / (P - 2)
    f3 = (1/(P-1))*(1/(P-2))

    denom_n = t1 - 2/P * t5 + (1/P)**2 * t9
    denom_s = P/(P-3) * (
        t1
        - f1 * t2
        + f2 * (2*t3 - t5)
        + f3 * (t6 - 2*t7 + t9)
    )
    denom_d = (P/(P-3))*(Q/(Q-1)) * (
        t1d
        - f1 * t2d
        + f2 * (2*t3d - t5d)
        + f3 * (t6d - 2*t7d + t9d)
    )

    numer_n = t6 - 2/P * t7 + (1/P)**2 * t9
    numer_s = P/(P-3) * (
        t6 
        - 2/(P-1) * t7 
        + 1/(P-2) * (4*t3 - P*t2) 
        + f3 * (t9 - 4*t5 + 2*t1 - t6) 
    )
    numer_d = (P/(P-3))*(Q/(Q-1))*(
        t6d 
        - 2/(P-1) * t7d 
        + 1/(P-2) * (4*t3d - P*t2d) 
        + f3 * (t9d - 4*t5d + 2*t1d - t6d) 
    )
    return denom_n, denom_s, denom_d, numer_n, numer_s, numer_d


In [15]:
from jax import random

# Example usage
key = random.PRNGKey(13)  # random seed

d=200

P=4
Q=8


Ke=jnp.power(10,jnp.linspace(0,-1,d))
cv=jnp.sqrt(Ke)

numer_true=jnp.sum(Ke)**2

denom_true=jnp.sum(Ke**2)
print(numer_true, denom_true, numer_true/denom_true)


numit=1000

out=[]
for i in range(numit):
    key0, key = jax.random.split(key)
    W=random.normal(key0, shape=(d, Q))
    key0, key = jax.random.split(key)
    X=random.normal(key0, shape=(P, d))
    key0, key = jax.random.split(key)
    ofs=random.normal(key0, shape=(1, Q))+1
    
    Phi=jnp.matmul(X*cv[None,:],W)+ofs
    
    out.append([getest_all(Phi,P,Q)])
out=np.squeeze(np.array(out))
    
mout=np.mean(out,axis=0)
denom=mout[2]
numer=mout[-1]

print(numer,denom, numer/denom)
#print(jnp.trace(jnp.matmul(Phi,Phi.T)/(P*Q))**2)


6136.06 43.287086 141.75267
6056.175 51.53946 117.50559
